# Function Testing Notebook

Author: Pete King

This notebook tests custom functions developed in the various helper modules to verify proper operation.

In [1]:
#123456789012345678901234567890123456789012345678901234567890123456789012345678
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import altair as alt
import yfinance as yf

import data_prep as dp

DATA_FILENAME='etf_raw_data.csv'
ETF='SPY'

# Deactivate the max rows and columns limit for Altair
alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

## Import and inspect ETF price data

In [2]:
df = pd.read_csv(
    DATA_FILENAME,
    index_col='date',
    parse_dates=True
)
df

,BIL,BND,GLD,HYG,IEF,IWM,LQD,QQQ,SPY,TIP,...,XLB,XLE,XLF,XLI,XLK,XLP,XLRE,XLU,XLV,XLY
date,,,,,,,,,,,,,,,,,,,,,
1993-01-29,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24.175381,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1993-02-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24.347315,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1993-02-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24.398914,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1993-02-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24.656811,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1993-02-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24.760000,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-03-25,91.580002,73.540001,416.290009,79.419998,95.360001,251.820007,108.730003,587.820007,656.820007,110.160004,...,49.410000,60.570000,49.340000,165.100006,136.759995,81.510002,40.270000,45.250000,146.240005,110.730003
2026-03-26,91.599998,73.110001,400.640015,78.919998,94.589996,247.440002,107.879997,573.789978,645.090027,109.760002,...,49.090000,61.520000,49.049999,161.270004,132.500000,81.139999,40.290001,45.330002,145.740005,108.830002
2026-03-27,91.629997,73.110001,414.700012,78.720001,94.599998,243.100006,107.620003,562.580017,634.090027,109.669998,...,48.910000,62.560001,47.810001,159.199997,129.919998,81.779999,40.009998,45.590000,143.259995,105.680000


## Compute and display daily return

Here we test the ability of the log_return function to compute daily returns and inspect the results.

In [3]:
test_df = df[[ETF]]
etf_return = test_df['SPY'].rolling(2).apply(dp.log_return, raw=True)
test_df[ETF + '_return'] = etf_return.values
test_df

,SPY,SPY_return
date,,
1993-01-29,24.175381,NaN
1993-02-01,24.347315,0.007087
1993-02-02,24.398914,0.002117
1993-02-03,24.656811,0.010515
1993-02-04,24.760000,0.004176
...,...,...
2026-03-25,656.820007,0.005557
2026-03-26,645.090027,-0.018020
2026-03-27,634.090027,-0.017199


In [4]:
chart = alt.Chart(test_df.dropna().reset_index()).mark_circle(size=10).encode(
    x='date:T',
    y=ETF + '_return:Q'
)
chart.properties(height=200, width=800)

alt.Chart(...)

## Discussion

From the chart we can see that the mean of daily returns appears to be nearly zero -- an empirical justification of the zero mean assumption for expected return (E\[R\]).

Since volatility for an asset is a measure of the deviation of returns from expected return, we can get a feel for an asset's volatility just by inspecting the plot.  We see a general trend of baseline low volatility (for example, from Jan 2004 to Jan 2007), with periods of high volatility that tend to gradually revert to baseline (for example during the 'Great Recession', from late 2007, spiking in late 2008 / early 2009, and gradually reverting to a lower baseline by roughly 2012).

In [5]:
etf_return.describe()

count    8348.000000
mean        0.000394
std         0.011726
min        -0.115887
25%        -0.004350
50%         0.000678
75%         0.005916
max         0.135577
Name: SPY, dtype: float64

The main idea with this project is to think of the daily return for each asset as a random variable (R), and then investigate its statistical properties.  

***Right away, from this simple statistical description (above), we can get an idea of what to expect for the properties of an asset's returns (R):***
 - Estimated **expected return** (E\[R\]): 0.04 percent (very close to zero)
 - Estimated long-term (baseline) **volatility**: 1.17 percent

*Note that the financial term "volatility" can have mean interpretations, but here we mean the long-term standard deviation of return (R), assuming zero mean.*

In [6]:
# Compute using the zero-mean assumption for expected return
vol = np.sqrt(
    np.sum(etf_return.dropna().values**2) / len(etf_return.dropna())
)
print(f'Estimated long-term volatility with zero-mean assumption: \
        {vol * 100:2.2f} percent'
     )

Estimated long-term volatility with zero-mean assumption:         1.17 percent
